# LeWM Training Only (Stable Push-T Flow)

This notebook mirrors the known-good `pusht_train.ipynb` training path.


In [ ]:
# 1) Training configuration
RUN_NAME = 'pusht'
DATA_CONFIG = 'pusht'
DATA_LOCAL = '/content/stable-wm/pusht_expert_train.h5'
MAX_EPOCHS = 3
BATCH_SIZE = 128
NUM_WORKERS = 2
PRECISION = 'bf16-mixed'
CHECKPOINT_EVERY_N_STEPS = 500
LEWM_REPO = 'https://github.com/lucas-maes/le-wm.git'
USE_WANDB = False


In [ ]:
# 2) Bootstrap: clone le-wm and fetch Push-T data
import os, subprocess
from pathlib import Path

STABLEWM_HOME = Path('/content/stable-wm')
STABLEWM_HOME.mkdir(parents=True, exist_ok=True)
os.environ['STABLEWM_HOME'] = str(STABLEWM_HOME)

if not os.path.exists('/content/le-wm'):
    subprocess.check_call(['bash','-lc', f'git clone --depth 1 {LEWM_REPO} /content/le-wm'])
else:
    print('/content/le-wm already exists')

if not os.path.exists(DATA_LOCAL):
    subprocess.check_call(['bash','-lc', 'pip install -q huggingface_hub zstandard'])
    py = r'''
from huggingface_hub import hf_hub_download
import zstandard, shutil
from pathlib import Path
out = Path('/content/stable-wm/pusht_expert_train.h5')
zst = Path('/content/stable-wm/pusht_expert_train.h5.zst')
dl = hf_hub_download(repo_id='quentinll/lewm-pusht', filename='pusht_expert_train.h5.zst', repo_type='dataset', local_dir='/content/stable-wm')
src = Path(dl)
if src != zst:
    shutil.copy2(src, zst)
dctx = zstandard.ZstdDecompressor()
with open(zst,'rb') as i, open(out,'wb') as o:
    dctx.copy_stream(i,o)
print('done', out, out.stat().st_size)
'''
    subprocess.check_call(['python3','-c', py])
else:
    print('Dataset already present:', DATA_LOCAL)


In [ ]:
# 3) Install dependencies
import subprocess
subprocess.check_call(['bash','-lc','python3 -m pip install -U pip setuptools wheel'])
subprocess.check_call(['bash','-lc',"pip install 'stable-worldmodel[train]' h5py einops matplotlib scikit-learn xvfbwrapper boto3 hydra-core lightning"])
print('Dependencies installed.')


In [ ]:
# 4) Environment checks
import os, subprocess
print('python:', subprocess.check_output(['python3','--version'], text=True).strip())
print('data exists:', os.path.exists(DATA_LOCAL), DATA_LOCAL)
print('data config:', DATA_CONFIG)
if not os.path.exists('/content/le-wm/train.py'):
    raise FileNotFoundError('Missing /content/le-wm/train.py even after bootstrap.')


In [ ]:
# 5) Launch training (stable path)
import os, subprocess
os.environ['STABLEWM_HOME'] = '/content/stable-wm'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
wandb_flag = '' if USE_WANDB else 'wandb.enabled=false'
cmd = (
    f"set -o pipefail; cd /content/le-wm && HYDRA_FULL_ERROR=1 python3 train.py "
    f"data={DATA_CONFIG} subdir={RUN_NAME} {wandb_flag} "
    f"trainer.max_epochs={MAX_EPOCHS} loader.batch_size={BATCH_SIZE} "
    f"loader.num_workers={NUM_WORKERS} loader.pin_memory=false loader.persistent_workers=false "
    f"trainer.precision={PRECISION} +checkpoint_every_n_steps={CHECKPOINT_EVERY_N_STEPS} "
    f"2>&1 | tee /content/train_log.txt"
)
print(cmd)
ret = subprocess.run(['bash','-lc',cmd])
if ret.returncode != 0:
    raise RuntimeError(f'Training failed with exit code {ret.returncode}. See /content/train_log.txt')
print('Training finished.')


In [ ]:
# 6) Locate checkpoints
import glob, os
cands = sorted(glob.glob(f'/content/stable-wm/{RUN_NAME}/*.ckpt'))
if not cands:
    print('No checkpoints found under', f'/content/stable-wm/{RUN_NAME}')
else:
    for p in cands[-20:]:
        print(p, os.path.getsize(p))


In [ ]:
# 7) Optional live monitor (run in separate cell while training is active)
# !tail -f /content/train_duckie.log
# !watch -n 30 'ls -lh /content/stable-wm/duckie_explore_retrain/*.ckpt 2>/dev/null || echo no-ckpt-yet'
